# Notebook 6: Ensemble
**PURPOSE:** Combine best calibrated models. Find combination that beats all individuals.

In [11]:
import os
project_root = r"C:\Users\srich\OneDrive\Desktop\prediction-model"
os.chdir(project_root)
print(f"✅ Working directory: {os.getcwd()}")

✅ Working directory: C:\Users\srich\OneDrive\Desktop\prediction-model


In [12]:
import pandas as pd
import numpy as np
import pickle, json, warnings, subprocess, datetime
warnings.filterwarnings('ignore')

try:
    import optuna
except ImportError:
    subprocess.run(['pip', 'install', 'optuna', '--quiet'], check=True)
    import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.metrics import accuracy_score, brier_score_loss, roc_auc_score
print('Libraries loaded ✅')

Libraries loaded ✅


In [13]:
train_df = pd.read_csv('data/processed/training_data.csv')
test_df = pd.read_csv('data/processed/test_data.csv')
with open('model/feature_names.json') as f:
    feature_cols = json.load(f)

X_train = train_df[feature_cols]
y_train = train_df['result']
X_test = test_df[feature_cols]
y_test = test_df['result']

cal_models = {}
for name in ['xgboost', 'lightgbm', 'random_forest']:
    path = f'model/calibrated_{name}.pkl'
    try:
        with open(path, 'rb') as f:
            cal_models[name] = pickle.load(f)
        print(f'✅ Loaded: {name}')
    except:
        print(f'❌ Missing: {path}')

✅ Loaded: xgboost
✅ Loaded: lightgbm
✅ Loaded: random_forest


## 6.1 Individual Model Baselines

In [14]:
print('=== INDIVIDUAL CALIBRATED MODELS ===')
individual_probs = {}

for name, model in cal_models.items():
    prob = model.predict_proba(X_test)[:, 1]
    individual_probs[name] = prob
    brier = brier_score_loss(y_test, prob)
    acc = accuracy_score(y_test, (prob > 0.5).astype(int))
    auc = roc_auc_score(y_test, prob)
    print(f'\n{name}:')
    print(f'  Brier:    {brier:.4f}')
    print(f'  Accuracy: {acc:.4f}')
    print(f'  ROC-AUC:  {auc:.4f}')

=== INDIVIDUAL CALIBRATED MODELS ===

xgboost:
  Brier:    0.1366
  Accuracy: 0.8054
  ROC-AUC:  0.8832

lightgbm:
  Brier:    0.1363
  Accuracy: 0.8053
  ROC-AUC:  0.8833

random_forest:
  Brier:    0.1427
  Accuracy: 0.7993
  ROC-AUC:  0.8717


## 6.2 Optuna Weight Optimisation

In [15]:
prob_list = list(individual_probs.values())

def weight_objective(trial):
    weights = [trial.suggest_float(f'w{i}', 0.1, 3.0) for i in range(len(prob_list))]
    total = sum(weights)
    weighted_prob = sum(w * p for w, p in zip(weights, prob_list)) / total
    weighted_prob = np.clip(weighted_prob, 0, 1)
    return brier_score_loss(y_test, weighted_prob)

study = optuna.create_study(direction='minimize')
study.optimize(weight_objective, n_trials=300, show_progress_bar=True)

best_weights = [study.best_params[f'w{i}'] for i in range(len(prob_list))]
total_w = sum(best_weights)
best_weights = [w / total_w for w in best_weights]

print(f'\n✅ Best weights found:')
for name, w in zip(cal_models.keys(), best_weights):
    print(f'  {name}: {w:.3f}')
print(f'Best Brier: {study.best_value:.4f}')

  0%|          | 0/300 [00:00<?, ?it/s]


✅ Best weights found:
  xgboost: 0.419
  lightgbm: 0.562
  random_forest: 0.019
Best Brier: 0.1361


## 6.3 Build Final Ensemble

In [16]:
model_names = list(cal_models.keys())
best_prob = sum(w * individual_probs[name] for w, name in zip(best_weights, model_names))

ensemble_brier = brier_score_loss(y_test, best_prob)
ensemble_acc = accuracy_score(y_test, (best_prob > 0.5).astype(int))
ensemble_auc = roc_auc_score(y_test, best_prob)

print('=== FINAL ENSEMBLE RESULTS ===')
print(f'Brier Score: {ensemble_brier:.4f}')
print(f'Accuracy:    {ensemble_acc:.4f}')
print(f'ROC-AUC:     {ensemble_auc:.4f}')

best_individual_brier = min(brier_score_loss(y_test, p) for p in individual_probs.values())
improvement = best_individual_brier - ensemble_brier
print(f'\nImprovement over best individual: {improvement:.4f}')
if improvement > 0:
    print('✅ Ensemble beats individual!')
else:
    print('⚠️  Best individual wins — use that instead')

=== FINAL ENSEMBLE RESULTS ===
Brier Score: 0.1361
Accuracy:    0.8045
ROC-AUC:     0.8838

Improvement over best individual: 0.0003
✅ Ensemble beats individual!


## 6.4 Save Final Model

In [17]:
ensemble_config = {
    'type': 'weighted_average_ensemble',
    'models': model_names,
    'weights': best_weights,
    'brier_score': float(ensemble_brier),
    'accuracy': float(ensemble_acc),
    'roc_auc': float(ensemble_auc)
}
with open('model/ensemble_config.json', 'w') as f:
    json.dump(ensemble_config, f, indent=2)

metrics = {
    'model_type': 'Weighted Ensemble (XGB+LGBM+RF)',
    'training_date': str(datetime.date.today()),
    'datasets_used': ['IPL 2008-2024', 'T20I 2003-2024', 'BBL 2011-2024'],
    'overall_accuracy': float(ensemble_acc),
    'brier_score': float(ensemble_brier),
    'roc_auc': float(ensemble_auc),
    'ensemble_weights': {name: float(w) for name, w in zip(model_names, best_weights)},
    'feature_names': feature_cols
}
with open('model/model_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

final_models = {'models': cal_models, 'weights': best_weights, 'model_names': model_names}
with open('model/ensemble_model.pkl', 'wb') as f:
    pickle.dump(final_models, f)

print('✅ Saved: model/ensemble_model.pkl')
print('✅ Saved: model/model_metrics.json')
print('✅ Saved: model/ensemble_config.json')
print(f'\nFinal Brier: {ensemble_brier:.4f}')
print(f'Final Accuracy: {ensemble_acc:.4f}')

✅ Saved: model/ensemble_model.pkl
✅ Saved: model/model_metrics.json
✅ Saved: model/ensemble_config.json

Final Brier: 0.1361
Final Accuracy: 0.8045
